# Restore saved Jev pilot in an authorized Colab account
No training. No locked-test evaluation. GPU access remains subject to Colab limits.
First make the private backup folder accessible in this account’s My Drive (a shared-folder shortcut also works). Set SOURCE_DIR below to that folder. The backup is read-only input; results are copied into a separate run folder.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shutil, zipfile
from google.colab import drive
import torch
assert torch.cuda.is_available(), 'A permitted GPU runtime is required. No training is needed.'
DRIVE_MOUNT = Path('/content/pilot-drive')
drive.mount(str(DRIVE_MOUNT))
assert os.path.ismount(DRIVE_MOUNT), 'Drive mount failed.'

# Change ONLY this path if the copied backup/shortcut has another name.
SOURCE_DIR = DRIVE_MOUNT/'MyDrive/SDT_JEV_Backup_20261110T190910Z'
RUN = DRIVE_MOUNT/'MyDrive/SDT_JEV_Behavior_Pilot_v2/20261110T190910Z'
assert SOURCE_DIR.is_dir(), f'Backup folder not found: {SOURCE_DIR}'
assert SOURCE_DIR.resolve() != RUN.resolve(), 'Source backup and working run must differ.'
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()
RUN.mkdir(parents=True, exist_ok=True)
def copy_once(source, target):
    if target.exists():
        return  # Never overwrite resumed outputs. Models/inputs are verified below.
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name+'.copying')
    shutil.copy2(source, partial)
    assert digest(partial) == digest(source), f'Copy mismatch: {source}'
    partial.replace(target)

# The separate backup stores each final model directly under its candidate folder.
# Restore the original model paths so recorded generation metadata remains valid.
for name in ['lr5e7', 'lr1e6']:
    source_model = SOURCE_DIR/name
    receipt = source_model/'weights-verified.json'
    assert receipt.is_file(), f'Missing receipt: {receipt}'
    for filename, expected in json.loads(receipt.read_text()).items():
        p = source_model/filename
        assert p.is_file() and p.stat().st_size == expected['bytes'], p
        assert digest(p) == expected['sha256'], f'Source hash mismatch: {p}'
    for p in source_model.iterdir():
        if p.is_file():
            target = RUN/name/p.name if p.name == 'weights-verified.json' else RUN/name/'model'/p.name
            copy_once(p, target)
for p in SOURCE_DIR.iterdir():
    if p.is_file(): copy_once(p, RUN/p.name)
for filename in ['pilot-pairs.jsonl', 'sample-report.json', 'sampling-audit.json', 'jev_behavior_pilot.zip']:
    assert (RUN/filename).is_file(), filename
    assert digest(RUN/filename) == digest(SOURCE_DIR/filename), f'Input mismatch: {filename}'

bundle = RUN/'jev_behavior_pilot.zip'
WORK = Path('/content/jev-behavior-pilot')
with zipfile.ZipFile(bundle) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(WORK)
os.chdir(WORK)
os.environ['PYTHONPATH'] = str(WORK/'src')
sys.path.insert(0, str(WORK/'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.5.3', 'trl==1.17.1', 'accelerate==1.15.0', 'datasets==4.8.5', '-e', '.'], check=True)
import yaml
print('Restored existing run:', RUN)
print('Source backup unchanged:', SOURCE_DIR)


## 1. Restore existing run and helpers


In [ ]:
assert os.path.ismount(DRIVE_MOUNT), 'Drive is not mounted.'
assert RUN.name == '20261110T190910Z' and RUN.is_dir()
PAIRS = RUN/'pilot-pairs.jsonl'
assert PAIRS.is_file()
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
EXPERIMENTS = [('lr5e7', 5e-7), ('lr1e6', 1e-6)]
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()
def run(name, module, *args):
    assert os.path.ismount(DRIVE_MOUNT), 'Drive disconnected.'
    assert module != 'sdt_dpo.train', 'Training is disabled in resume mode.'
    with (RUN/(name+'.log')).open('a') as log:
        p = subprocess.Popen([sys.executable, '-u', '-m', module, *map(str,args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end=''); log.write(line); log.flush()
        if p.wait() != 0: raise RuntimeError(f'{name} failed; inspect its saved log.')
print('Existing run selected. Training is disabled:', RUN)


## 2. Verify both saved models (no training)


In [ ]:
# Training already completed. Verify saved models; NEVER retrain here.
for name, _ in EXPERIMENTS:
    model = RUN/name/'model'
    receipt = RUN/name/'weights-verified.json'
    assert (model/'config.json').is_file() and receipt.is_file(), name
    expected_files = json.loads(receipt.read_text())
    assert expected_files, 'Empty weight receipt'
    for filename, expected in expected_files.items():
        path = model/filename
        assert path.is_file() and path.stat().st_size == expected['bytes'], f'Missing/incomplete weights: {path}'
        assert digest(path) == expected['sha256'], f'Weight hash mismatch: {path}'
    print('Verified saved model:', name)


## 3. Validation comparison and generated answers
Fixed pairs diagnose preference learning. Generated answers are needed to assess actual SDT behavior. All models use identical decoding; no SDT system-prompt advantage.

In [ ]:
# Reuse complete pair evaluations; generation resumes by prompt ID.
from sdt_dpo.generate import load_prompts
expected_prompts = {r['prompt_id']: r['prompt'] for r in load_prompts(PAIRS, split='validation', min_margin=0.10)}
expected_pair_ids = {r['pair_id'] for r in map(json.loads, PAIRS.read_text().splitlines()) if r['split']=='validation'}
assert len(expected_prompts)==200 and len(expected_pair_ids)==583
for name, model in [('baseline', MODEL)] + [(n, str(RUN/n/'model')) for n,_ in EXPERIMENTS]:
    summary = RUN/(name+'-validation.json')
    details = RUN/(name+'-validation-pairs.jsonl')
    complete = False
    if summary.exists() and details.exists():
        report = json.loads(summary.read_text())
        rows = [json.loads(line) for line in details.read_text().splitlines() if line.strip()]
        complete = (report.get('model') == model and report.get('split') == 'validation'
                    and report.get('overall',{}).get('n') == len(expected_pair_ids)
                    and len(rows) == len(expected_pair_ids)
                    and {r['pair_id'] for r in rows} == expected_pair_ids)
    if complete:
        print('Reusing complete fixed-pair evaluation:', name)
    else:
        run(name+'-validation', 'sdt_dpo.evaluate', '--pairs', PAIRS, '--model', model,
            '--split','validation','--max-length','1024','--output',summary,'--details',details)
    generations = RUN/(name+'-validation-generations.jsonl')
    run(name+'-validation-generation', 'sdt_dpo.generate', '--pairs',PAIRS,'--model',model,
        '--split','validation','--max-new-tokens','512','--temperature','0','--output',generations)
    rows = [json.loads(line) for line in generations.read_text().splitlines() if line.strip()]
    assert len(rows)==len(expected_prompts) and {r['prompt_id'] for r in rows}==set(expected_prompts)
    assert all(r['prompt']==expected_prompts[r['prompt_id']] and r['response'].strip() for r in rows)
    print('Verified all 200 responses:', name)
    if name != 'baseline':
        run(name+'-validation-comparison','sdt_dpo.compare','--baseline-details',RUN/'baseline-validation-pairs.jsonl',
            '--dpo-details',details,'--output',RUN/(name+'-validation-comparison.json'))


## 4. Prepare anonymous responses for JEV
Send only the blinded file to the evaluator; keep the mapping private. Agree with Ghadi on the exact JEV prompt/schema and quality rule. Choose a candidate using validation SDT and quality results, not test results. Neither longer answers nor higher DPO implicit accuracy alone establishes success.

In [ ]:
import random
def export_blind(split, names):
    rows, mapping, expected_ids = [], {}, None
    for name in names:
        source = RUN/(name+'-'+split+'-generations.jsonl')
        records = [json.loads(line) for line in source.read_text().splitlines() if line.strip()]
        ids = {r['prompt_id'] for r in records}
        assert len(ids) == len(records)
        if expected_ids is None: expected_ids = ids
        else: assert ids == expected_ids, 'Generation sets differ'
        for r in records:
            key = hashlib.sha256((split+name+r['prompt_id']).encode()).hexdigest()[:24]
            # The repository generation schema stores response text in response.
            text = r['response']
            assert text.strip()
            rows.append(dict(id=key, prompt=r['prompt'], response=text))
            mapping[key] = dict(model=name, prompt_id=r['prompt_id'])
    random.Random(42).shuffle(rows)
    (RUN/(split+'-jev-blinded.jsonl')).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows))
    (RUN/(split+'-jev-private-map.json')).write_text(json.dumps(mapping,indent=2))
    print('Prepared',len(rows),'responses for',split)
export_blind('validation', ['baseline']+[n for n,_ in EXPERIMENTS])


## 5. Freeze selection, then unlock the test once
Leave disabled until validation scoring is reviewed. Set the reason before opening test results. Do not retune on this test. A quality drop may justify selecting neither candidate.

In [ ]:
RUN_LOCKED_TEST = False
SELECTED_NAME = ''
SELECTION_REASON = ''
if RUN_LOCKED_TEST:
    assert SELECTED_NAME in dict(EXPERIMENTS) and SELECTION_REASON.strip()
    selected = dict(name=SELECTED_NAME, reason=SELECTION_REASON)
    lock = RUN/'selected-configuration.json'
    if lock.exists(): assert json.loads(lock.read_text()) == selected, 'Test selection is already frozen'
    else: lock.write_text(json.dumps(selected,indent=2))
    for name, model in [('baseline',MODEL),(SELECTED_NAME,str(RUN/SELECTED_NAME/'model'))]:
        run(name+'-test','sdt_dpo.evaluate','--pairs',PAIRS,'--model',model,'--split','test',
            '--max-length','1024','--output',RUN/(name+'-test.json'),'--details',RUN/(name+'-test-pairs.jsonl'))
        run(name+'-test-generation','sdt_dpo.generate','--pairs',PAIRS,'--model',model,'--split','test',
            '--max-new-tokens','512','--temperature','0','--output',RUN/(name+'-test-generations.jsonl'))
    run('test-comparison','sdt_dpo.compare','--baseline-details',RUN/'baseline-test-pairs.jsonl',
        '--dpo-details',RUN/(SELECTED_NAME+'-test-pairs.jsonl'),'--output',RUN/'test-comparison.json')
    export_blind('test',['baseline',SELECTED_NAME])
else:
    print('Locked test not opened. Finish validation JEV scoring first.')


## 6. Verify Drive persistence before disconnecting

In [ ]:
drive.flush_and_unmount()
drive.mount(str(DRIVE_MOUNT))
assert os.path.ismount(DRIVE_MOUNT), 'Drive remount failed.'
for name,_ in EXPERIMENTS:
    model = RUN/name/'model'
    receipt = RUN/name/'weights-verified.json'
    assert receipt.exists(), name
    for filename, expected in json.loads(receipt.read_text()).items():
        assert (model/filename).stat().st_size == expected['bytes']
        assert digest(model/filename) == expected['sha256']
print('PASS: saved model weights verified after Drive remount.',RUN)
